In [2]:
import pandas as pd

In [5]:
ratings_df = pd.read_csv('../data/raw/ml-25m/ratings.csv')
print(ratings_df.head())

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510


In [7]:
# We have 162541 users in the dataset
ratings_df["userId"].nunique()

162541

In [6]:
# Number of ratings submitted by each user
ratings_per_user = ratings_df.groupby("userId").size().reset_index(name="rating_count")
ratings_per_user

,userId,rating_count
0,1,70
1,2,184
2,3,656
3,4,242
4,5,101
...,...,...
162536,162537,101
162537,162538,154
162538,162539,47
162539,162540,88


In [9]:
# Number of ratings for each movie

ratings_per_movie = ratings_df.groupby("movieId").size().reset_index(name="movie_rating_count")
ratings_per_movie

,movieId,movie_rating_count
0,1,57309
1,2,24228
2,3,11804
3,4,2523
4,5,11714
...,...,...
59042,209157,1
59043,209159,1
59044,209163,1
59045,209169,1


In [10]:
# Calculate the distribution of rating values
rating_distribution = (
    ratings_df["rating"]
    .value_counts()
    .sort_index()
    .reset_index()
)

rating_distribution.columns = ["rating", "count"]
rating_distribution

,rating,count
0,0.5,393068
1,1.0,776815
2,1.5,399490
3,2.0,1640868
4,2.5,1262797
5,3.0,4896928
6,3.5,3177318
7,4.0,6639798
8,4.5,2200539
9,5.0,3612474


In [11]:
movies_df = pd.read_csv('../data/raw/ml-25m/movies.csv')
print(movies_df.head())

# Explode genres (they are pipe-separated) and calculate distribution
genre_distribution = (
    movies_df['genres']
    .str.split('|')
    .explode()
    .value_counts()
    .reset_index()
)

genre_distribution.columns = ['genre', 'count']
genre_distribution

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


,genre,count
0,Drama,25606
1,Comedy,16870
2,Thriller,8654
3,Romance,7719
4,Action,7348
5,Horror,5989
6,Documentary,5605
7,Crime,5319
8,(no genres listed),5062
9,Adventure,4145


In [12]:
# Convert Unix timestamp to a proper datetime column for each rating interaction
ratings_df["interaction_timestamp"] = pd.to_datetime(ratings_df["timestamp"], unit="s", utc=True)

# Quick validation
ratings_df[["timestamp", "interaction_timestamp"]].head()

,timestamp,interaction_timestamp
0,1147880044,2006-05-17 15:34:04+00:00
1,1147868817,2006-05-17 12:26:57+00:00
2,1147868828,2006-05-17 12:27:08+00:00
3,1147878820,2006-05-17 15:13:40+00:00
4,1147868510,2006-05-17 12:21:50+00:00


In [13]:
top_5_movies = (
    ratings_per_movie
    .merge(movies_df[["movieId", "title"]], on="movieId", how="left")
    .sort_values("movie_rating_count", ascending=False)
    .head(5)
    [["movieId", "title", "movie_rating_count"]]
)

top_5_movies

,movieId,title,movie_rating_count
351,356,Forrest Gump (1994),81491
314,318,"Shawshank Redemption, The (1994)",81482
292,296,Pulp Fiction (1994),79672
585,593,"Silence of the Lambs, The (1991)",74127
2480,2571,"Matrix, The (1999)",72674


In [14]:
# Calculate sparsity
total_possible_interactions = ratings_df["userId"].nunique() * ratings_df["movieId"].nunique()
actual_interactions = len(ratings_df)
sparsity = (1 - (actual_interactions / total_possible_interactions)) * 100

print(f"Total possible interactions: {total_possible_interactions:,}")
print(f"Actual interactions: {actual_interactions:,}")
print(f"Sparsity: {sparsity:.2f}%")

Total possible interactions: 9,597,558,427
Actual interactions: 25,000,095
Sparsity: 99.74%


In [16]:
count_35_and_above = (ratings_df["rating"] >= 4.0).sum()
count_below_35 = (ratings_df["rating"] < 4.0).sum()

print(f"Ratings 4.0 and above: {count_35_and_above:,}")
print(f"Ratings below 4.0: {count_below_35:,}")

Ratings 4.0 and above: 12,452,811
Ratings below 4.0: 12,547,284
